In [ ]:
import os
import glob
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# -----------------------------
# Hyperparameters
# -----------------------------
DATA_PATH = "/content/drive/My Drive/MAJOR PCOS/pcos_rotterdam_balanceado.csv"

N_CLIENTS = 10
COMMUNICATION_ROUNDS = 20

# Stability-focused choices
CLIENT_EPOCHS = 2              # reduce local epochs to avoid overfitting
CLIENT_BATCH_SIZE = 128        # larger batch for smoother updates
LEARNING_RATE = 1e-4

# PSO settings (conservative)
PSO_PARTICLES = 8
PSO_ITERS = 2
PSO_W = 0.6
PSO_C1 = 1.2
PSO_C2 = 1.2
PSO_BOUND_DELTA = 0.01   # particles constrained to start +/- this amount

SEED = 42
VERBOSE = 1

tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [ ]:
print("Using dataset:", DATA_PATH)
df = pd.read_csv(DATA_PATH)
print("Raw dataset shape:", df.shape)

LABEL_COL = df.columns[-1]
X = df.drop(columns=[LABEL_COL]).values
y = df[LABEL_COL].values

# Encode labels if needed
if y.dtype == object or y.dtype == str:
    y = LabelEncoder().fit_transform(y)

# Standardize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X, y)

# Check class balance after SMOTE
print("Class distribution after SMOTE:\n", pd.Series(y_smote).value_counts())

# assume binary classification as your create_mlp uses sigmoid
# split into train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=SEED, stratify=y
)

print("Train/Test shapes:", X_train.shape, X_test.shape)

Using dataset: /content/drive/My Drive/MAJOR PCOS/pcos_rotterdam_balanceado.csv
Raw dataset shape: (3000, 6)
Class distribution after SMOTE:
 0    2400
1    2400
Name: count, dtype: int64
Train/Test shapes: (2700, 5) (300, 5)


In [ ]:
# # -----------------------------
# # Load Dataset
# # -----------------------------
# df = pd.read_csv(DATA_PATH)
# LABEL_COL = df.columns[-1]
# X = df.drop(columns=[LABEL_COL]).values
# y = df[LABEL_COL].values

# if y.dtype == object or y.dtype == str:
#     y = LabelEncoder().fit_transform(y)

# scaler = StandardScaler()
# X = scaler.fit_transform(X)

# from imblearn.over_sampling import SMOTE

# smote = SMOTE(random_state=42)
# X_smote, y_smote = smote.fit_resample(X, y)

# # Check class balance after SMOTE
# print("Class distribution after SMOTE:\n", pd.Series(y_smote).value_counts())

# is_classification = (len(np.unique(y)) <= 50)

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.1, random_state=SEED, stratify=y if is_classification else None
# )

Class distribution after SMOTE:
 0    2400
1    2400
Name: count, dtype: int64


In [ ]:
# -----------------------------
# Partition training data among clients
# -----------------------------
def create_clients(X, y, n_clients):
    idx = np.arange(len(X))
    np.random.shuffle(idx)
    splits = np.array_split(idx, n_clients)
    return [(X[s], y[s]) for s in splits]

clients = create_clients(X_train, y_train, N_CLIENTS)
print(f"Created {len(clients)} clients. Example client sizes:", [len(c[0]) for c in clients])


Created 10 clients. Example client sizes: [270, 270, 270, 270, 270, 270, 270, 270, 270, 270]


In [ ]:
from tensorflow.keras.layers import Input
# MLP Model
def create_mlp(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)), # Explicit Input layer to avoid warning
        Dense(128, activation="relu"),
        Dropout(0.2),
        Dense(64, activation="relu"),
        Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss=BinaryCrossentropy(),
        metrics=["accuracy"]
    )
    return model

In [ ]:
# -----------------------------
# Utility: flatten/unflatten weights
# -----------------------------
def flatten_weights(weights_list):
    flat = np.concatenate([w.flatten() for w in weights_list])
    shapes = [w.shape for w in weights_list]
    sizes = [w.size for w in weights_list]
    return flat, shapes, sizes

def unflatten_weights(flat, shapes, sizes):
    out = []
    idx = 0
    for shp, sz in zip(shapes, sizes):
        out.append(flat[idx: idx + sz].reshape(shp))
        idx += sz
    return out

In [ ]:
# Build initial global model to get shapes
INPUT_DIM = X_train.shape[1]
global_model = create_mlp(INPUT_DIM)
global_weights = global_model.get_weights()
flat_w, shapes, sizes = flatten_weights(global_weights)
D = flat_w.size
print("Flattened weights dimension:", D)

Flattened weights dimension: 9089


In [ ]:
# -----------------------------
# PSO fitness: average client TRAIN loss (we do not use validation)
# -----------------------------
def make_fitness_fn(model_template, shapes, sizes, clients):
    # clone model architecture (weights will be set from vector)
    eval_model = tf.keras.models.clone_model(model_template)
    eval_model.compile(optimizer='adam', loss='binary_crossentropy')

    def f(flat_vec):
        weights = unflatten_weights(flat_vec, shapes, sizes)
        eval_model.set_weights(weights)
        losses = []
        for Xc, yc in clients:
            # use evaluate in verbose=0 to get loss on each client's local data
            loss = eval_model.evaluate(Xc, yc, verbose=0)[0]
            losses.append(loss)
        return float(np.mean(losses))
    return f

fitness = make_fitness_fn(global_model, shapes, sizes, clients)

In [ ]:
# -----------------------------
# PSO method with bounded search around `start`
# -----------------------------
def pso_optimize(start):
    # start: flat vector (numpy)
    lb = start - PSO_BOUND_DELTA
    ub = start + PSO_BOUND_DELTA

    # initialize particles uniformly in the bounds
    particles = np.array([np.random.uniform(lb, ub) for _ in range(PSO_PARTICLES)])
    velocities = np.zeros_like(particles)
    pbest = particles.copy()
    pbest_vals = np.array([fitness(p) for p in pbest])
    gbest_idx = int(np.argmin(pbest_vals))
    gbest = pbest[gbest_idx].copy()
    gbest_val = pbest_vals[gbest_idx]

    for it in range(PSO_ITERS):
        for i in range(PSO_PARTICLES):
            r1 = np.random.rand(D)
            r2 = np.random.rand(D)
            velocities[i] = PSO_W * velocities[i] + PSO_C1 * r1 * (pbest[i] - particles[i]) + PSO_C2 * r2 * (gbest - particles[i])
            particles[i] = particles[i] + velocities[i]
            # enforce bounds
            np.clip(particles[i], lb, ub, out=particles[i])

            val = fitness(particles[i])
            if val < pbest_vals[i]:
                pbest_vals[i] = val
                pbest[i] = particles[i].copy()
                if val < gbest_val:
                    gbest_val = val
                    gbest = particles[i].copy()

        if VERBOSE:
            print(f"   PSO iter {it+1}/{PSO_ITERS} best_loss={gbest_val:.6f}")

    return gbest

In [ ]:
# -----------------------------
# Federated training with FedPSO (clean printing + final save only)
# -----------------------------
log_rows = []

print("\n🚀 Starting FedPSO Training")

for r in range(1, COMMUNICATION_ROUNDS + 1):
    print(f"\n🔁 Round {r}/{COMMUNICATION_ROUNDS}")

    client_weights = []

    # local training on each client
    for (Xc, yc) in clients:
        local = create_mlp(INPUT_DIM)
        local.set_weights(global_weights)
        es = EarlyStopping(monitor='loss', patience=1, restore_best_weights=True, verbose=0)
        local.fit(Xc, yc, epochs=CLIENT_EPOCHS, batch_size=CLIENT_BATCH_SIZE, verbose=0, callbacks=[es])
        client_weights.append(local.get_weights())

    # aggregation start = mean
    flat_clients = np.array([flatten_weights(w)[0] for w in client_weights])
    start = flat_clients.mean(axis=0)

    # PSO refinement
    best_flat = pso_optimize(start)

    # update global model
    global_weights = unflatten_weights(best_flat, shapes, sizes)
    global_model.set_weights(global_weights)

    # evaluate on test set
    test_loss, test_acc = global_model.evaluate(X_test, y_test, verbose=0)
    print(f"   📌 Loss: {test_loss:.6f}  |  Accuracy: {test_acc:.4f}")

    log_rows.append({
        "round": r,
        "test_loss": float(test_loss),
        "test_acc": float(test_acc)
    })


🚀 Starting FedPSO Training

🔁 Round 1/20
   PSO iter 1/2 best_loss=0.641696
   PSO iter 2/2 best_loss=0.638430
   📌 Loss: 0.639042  |  Accuracy: 0.9067

🔁 Round 2/20
   PSO iter 1/2 best_loss=0.614470
   PSO iter 2/2 best_loss=0.613296
   📌 Loss: 0.614115  |  Accuracy: 0.9433

🔁 Round 3/20
   PSO iter 1/2 best_loss=0.587976
   PSO iter 2/2 best_loss=0.584666
   📌 Loss: 0.584889  |  Accuracy: 0.9767

🔁 Round 4/20
   PSO iter 1/2 best_loss=0.566173
   PSO iter 2/2 best_loss=0.562006
   📌 Loss: 0.561392  |  Accuracy: 0.9900

🔁 Round 5/20
   PSO iter 1/2 best_loss=0.542183
   PSO iter 2/2 best_loss=0.540812
   📌 Loss: 0.539856  |  Accuracy: 0.9900

🔁 Round 6/20
   PSO iter 1/2 best_loss=0.521795
   PSO iter 2/2 best_loss=0.518180
   📌 Loss: 0.516703  |  Accuracy: 0.9967

🔁 Round 7/20
   PSO iter 1/2 best_loss=0.497586
   PSO iter 2/2 best_loss=0.493362
   📌 Loss: 0.491084  |  Accuracy: 0.9933

🔁 Round 8/20
   PSO iter 1/2 best_loss=0.474034
   PSO iter 2/2 best_loss=0.472652
   📌 Loss: 0.

In [ ]:
OUTPUT_WEIGHTS_FILE = "/content/drive/MyDrive/MAJOR PCOS/MLP FedPSO (CSV).keras"
global_model.save(OUTPUT_WEIGHTS_FILE)
print(f"\n✅ PSO-FedAvg complete! Model saved to {OUTPUT_WEIGHTS_FILE}")


✅ PSO-FedAvg complete! Model saved to /content/drive/MyDrive/MAJOR PCOS/MLP FedPSO (CSV).keras


In [ ]:
best_model_path =

# save final model only (not every round)
global_model.save(best_model_path)

# save logs to CSV
log_df = pd.DataFrame(log_rows)
log_csv_path = os.path.join(DATA_DIR, "fedpso_rounds_log.csv")
log_df.to_csv(log_csv_path, index=False)

print("\n🏁 FedPSO Training Completed.")
print("Final model saved at:", best_model_path)
print("Training logs saved at:", log_csv_path)


In [ ]:
# -----------------------------
# Final Test
# -----------------------------
print("\n📌 Final Test Evaluation")
test_results = global_model.evaluate(X_test, y_test, verbose=1)
print("🔍 Test Metrics:", test_results)

# -----------------------------
# Save Model
# -----------------------------
out_path = "/mnt/data/fedpso_mlp_final.h5"
global_model.save(out_path)
print("\n💾 Model saved to:", out_path)